# Reduced-Order Finger-Actuator Parameter Map

Purpose: keep the short-term Python model and the parametric Fusion model aligned. This is a working design-spec notebook, not a final validated model.

Current model status:

- Implemented direction: 2D planar index finger, three rigid phalanges, MCP/PIP/DIP revolute joints, effective tendon routing points, tendon path length, tendon excursion, fingertip position, and local $\partial L / \partial q$.
- Correct current use: compare routing geometry, ring placement, tendon stroke, and torque-per-tendon-tension before committing to hardware.
- Not solved yet: dynamic speed/rhythm, closed-loop trajectory tracking, friction, slack/hysteresis, contact mechanics, and motor selection with real efficiency/current limits.
- Sign convention in the current v0 code: zero angles are a straight finger along `+x`; negative joint angles correspond to physiological flexion for the current palmar routing choice.


## 1. State Variables And Geometry

| Group | Symbol / parameter | Meaning | Python role | Fusion role |
|---|---|---|---|---|
| Joint state | $q = [\theta_{\mathrm{MCP}}, \theta_{\mathrm{PIP}}, \theta_{\mathrm{DIP}}]^T$ | Relative joint angles | model state, sweep variable, measured output | optional pose parameters for assemblies |
| Joint velocity | $\dot{q}$ | Joint angular velocity | needed for speed, rhythm, and actuator velocity | not required for static CAD |
| Link lengths | $l_1, l_2, l_3$ | proximal, middle, distal phalanx lengths | forward kinematics | main skeletal dimensions |
| Link section | $w_i, t_i$ | local finger width/thickness | ring clearance and offset estimate | ring inner diameter and clearance |
| Joint limits | $q_{\min}, q_{\max}$ | allowable ROM per joint | sweep bounds and validity checks | motion-study limits if used |
| Neutral/rest pose | $q_0$ | passive zero-load reference | passive torque law | optional neutral assembly pose |
| Joint coupling | $\theta_{\mathrm{DIP}} = \alpha\,\theta_{\mathrm{PIP}}$ | optional anatomical simplification | reduce DOF or add constraint | not a direct CAD parameter |

Forward kinematics for the planar index finger:

$$
\begin{aligned}
\phi_1 &= \theta_{\mathrm{MCP}} \\
\phi_2 &= \theta_{\mathrm{MCP}} + \theta_{\mathrm{PIP}} \\
\phi_3 &= \theta_{\mathrm{MCP}} + \theta_{\mathrm{PIP}} + \theta_{\mathrm{DIP}}
\end{aligned}
$$

$$
\begin{aligned}
p_{\mathrm{tip}}(q) &= p_{\mathrm{MCP}}
+ l_1
\begin{bmatrix}
\cos \phi_1 \\
\sin \phi_1
\end{bmatrix}
+ l_2
\begin{bmatrix}
\cos \phi_2 \\
\sin \phi_2
\end{bmatrix}
+ l_3
\begin{bmatrix}
\cos \phi_3 \\
\sin \phi_3
\end{bmatrix}
\end{aligned}
$$

Fingertip position should be a primary model output, because it connects joint-space simulation to pinch/tapping experiments and marker tracking.


## 2. Routing, Rings, Branches, And Thimble Parameters

| Group | Symbol / parameter | Meaning | Python role | Fusion role |
|---|---|---|---|---|
| Independent tension input | $u_{T,\mathrm{flex}}, u_{T,\mathrm{ext}}$ | actuator-side tension command or tendon drive input | one scalar input can drive one or more branches | actuator/TSA/spool assignment |
| Routing branch | $\mathrm{branch}_{j}$ | one geometric tendon path sharing an input tension | one `RoutingPath` or branch path | one side route, tube, slot, or guide sequence |
| Branch count per input | $N_{\mathrm{branch}}$ | number of geometric branches driven by the same input | e.g. left/right side branches with shared tension | left/right sleeve or ring layouts |
| Routing path family | $\mathrm{path}_{\mathrm{flex}},\ \mathrm{path}_{\mathrm{ext}}$ | grouped branches for flexion or extension | flexion and extension inputs are modeled separately | separate hole/slot layouts |
| Entry or splitter point | $p_{\mathrm{entry}}, p_{\mathrm{split}}$ | actuator-side entry and optional branch split/yoke point | first point or branch common boundary | wrist/palm guide or splitter geometry |
| Ring / guide count | $N_{\mathrm{guides}}$ | number of physical or effective guides per branch | design variable to sweep | pattern count or explicit components |
| Ring / guide placement | $u_{\mathrm{guide}},\ v_{\mathrm{guide}}$ | longitudinal fraction and side/palmar/dorsal offset on a segment | routing-point coordinates | sketch dimensions or user parameters |
| Ring inner diameter | $D_{\mathrm{ring,inner}}$ | clearance around finger | constraint, not usually solved by physics | direct CAD parameter |
| Ring wall thickness | $t_{\mathrm{ring,wall}}$ | stiffness/strength/printability | later fabrication constraint | direct CAD parameter |
| Tendon slot/hole | $D_{\mathrm{slot}},\ \mathrm{slot}_{\mathrm{offset}}$ | tendon clearance and line of action | effective guide offset | direct CAD parameter |
| Thimble geometry | $D_{\mathrm{thimble,inner}},\ t_{\mathrm{thimble,wall}}$ | fingertip interface dimensions | mostly boundary condition | direct CAD parameter |
| Thimble anchor | $p_{\mathrm{anchor}}$ | terminal tendon attachment point | final routing point on a branch | direct CAD parameter |
| Tendon diameter | $D_{\mathrm{tendon}}$ | cord/wire diameter | friction/clearance later | slot sizing and bend-radius check |
| Initial slack | $s_{0,\mathrm{flex}},\ s_{0,\mathrm{ext}}$ | slack/pretension state | needed before equilibrium/control | assembly/tensioning detail |

Important distinction: an **independent tension input** is not the same thing as a physical routing branch. The intended v0 architecture can have one flexion input driven by one actuator or TSA, while that input splits into two side branches around the finger. In that case the branches share one scalar tension in the first model.

A routing element in Python is an effective point. It does not always mean one physical ring. One physical ring, slot, or tube may create one or more effective contact/guide points, and one effective guide may later be implemented with a slot, ring, thimble feature, or low-friction tube.

For v0, side-routed tendons or cords through soft/open guide elements should be represented as effective routing geometry, not as literal tubing material. Guide material, tube compliance, ring deformation, friction, and local pressure enter later through friction, slack, hysteresis, guide-compliance, or comfort measurements.

Ring diameter is mostly a CAD/comfort constraint. Ring placement, branch count, guide count, and entry/splitter geometry become model outputs only after an optimization or ranking criterion is defined.



## 3. Tendon Length, Excursion, Branches, And Moment Arms

For each tendon branch, define world-frame routing points $p_i(q)$ and total branch path length:

$$
\begin{aligned}
L_j(q) &= \sum_i \left\| p_{i+1,j}(q) - p_{i,j}(q) \right\| \\
\Delta L_j(q) &= L_j(q) - L_j(q_{\mathrm{ref}}) \\
g_j(q) &= \frac{\partial L_j}{\partial q}
\end{aligned}
$$

where $j$ indexes a routing branch, such as left-side and right-side branches driven by the same flexion input.

For a single branch, the virtual-work relationship between branch tension and generalized joint torque is:

$$
\tau_{\mathrm{branch},j}(q,T_j) = -T_j\frac{\partial L_j(q)}{\partial q}
$$

For one independent input that splits into two branches and is assumed to share one tension $T$, the first v0 approximation is:

$$
\begin{aligned}
R_j(q) &= -\frac{\partial L_j(q)}{\partial q} \\
R_{\mathrm{total}}(q) &= \sum_j R_j(q) \\
\tau_{\mathrm{tendon}}(q,T) &= T R_{\mathrm{total}}(q)
\end{aligned}
$$

This means the torque contribution from the left and right side branches adds, while the actuator still has only one independent tension input.

For the current palmar/flexion routing, flexion can shorten a branch path. In that case actuator pull length is better written as:

$$
s_{\mathrm{pull,flex}}(q) = L_{\mathrm{flex}}(q_{\mathrm{ref}}) - L_{\mathrm{flex}}(q)
$$

rather than assuming $\Delta L$ is always positive. The sign matters when converting model stroke into spool rotation or TSA contraction.

For symmetric left/right branches, v0 can approximate actuator pull using one representative branch pull. For asymmetric branches, branch slack and splitter/yoke behavior become real mechanism variables and should be measured or modeled explicitly.

$\partial L / \partial q$ has units of meters per radian, so it behaves like a posture-dependent moment arm. Because the current geometry code uses millimeters, torque-per-newton values must be converted from `N*mm/N` to `N*m/N` by multiplying by `1e-3`.

## 4. Passive Joint Torque Laws

Start with a linear passive law because it is interpretable and easy to identify from simple bench data:

$$
\tau_{\mathrm{passive},i}(q_i) = -k_{1,i}(q_i - q_{0,i})
$$

The minus sign means passive torque resists displacement away from the rest pose. Some code may separately report

$$
\tau_{\mathrm{required},i}(q_i) = k_{1,i}(q_i - q_{0,i})
$$

which is the external torque required to hold that displacement.

Nonlinear option for larger ROM or if experiments show clear stiffening near end range:

$$
\begin{aligned}
\tau_{\mathrm{passive},i}(q_i)
&= -\left[k_{1,i}(q_i - q_{0,i}) + k_{3,i}(q_i - q_{0,i})^3\right] \\
K_{\mathrm{local},i}(q_i)
&= k_{1,i} + 3k_{3,i}(q_i - q_{0,i})^2
\end{aligned}
$$

Recommendation: keep linear as the baseline, but include the cubic option in code. Nonlinear passive torque is not needed to compare first-pass routing geometries, but it may be needed once equilibrium predictions are compared to physical motion.


## 5. Quasi-Static Equilibrium And Required Tension

For free-space quasi-static motion, solve a torque residual:

$$
\begin{aligned}
\tau_{\mathrm{res}}(q, T_{\mathrm{flex}}, T_{\mathrm{ext}})
&= \tau_{\mathrm{tendon,flex}}(q, T_{\mathrm{flex}})
+ \tau_{\mathrm{tendon,ext}}(q, T_{\mathrm{ext}}) \\
&\quad + \tau_{\mathrm{passive}}(q)
+ \tau_{\mathrm{gravity}}(q)
+ \tau_{\mathrm{contact}}(q)
\end{aligned}
$$

$$
\tau_{\mathrm{res}} = 0
$$

Early model simplification:

$$
\tau_{\mathrm{gravity}} = 0, \qquad
\tau_{\mathrm{contact}} = 0, \qquad
F_{\mathrm{friction}} = 0
$$

Required tendon tension is not generally $\tau_{\mathrm{tendon}} / \lvert \partial L / \partial q \rvert$ for the full 3-joint finger. With one independent tension input and three joints, the system is underactuated: one scalar tension creates a 3D joint-torque vector shaped by the total routing moment-arm map.

For one flexion input with one or more branches sharing the same tension, define:

$$
a(q) = R_{\mathrm{total}}(q) = \sum_j -\frac{\partial L_j(q)}{\partial q}
$$

For a desired or required assistive torque vector $\tau_{\mathrm{req}}$, the best one-input least-squares estimate is:

$$
\begin{aligned}
T_{\mathrm{req}} &= \frac{a(q)^T\tau_{\mathrm{req}}}{a(q)^Ta(q)} \\
\tau_{\mathrm{fit}} &= T_{\mathrm{req}}a(q) \\
\tau_{\mathrm{error}} &= \tau_{\mathrm{req}} - \tau_{\mathrm{fit}}
\end{aligned}
$$

Then clamp or reject cases with $T_{\mathrm{req}} < 0$, because a tendon can pull but not push. The remaining $\tau_{\mathrm{error}}$ is the underactuation mismatch: it tells whether the routing can produce the desired joint torque distribution.

For a single-joint proxy, the simpler relationship is acceptable:

$$
T_{\mathrm{req}} = -\frac{\tau_{\mathrm{req}}}{\partial L / \partial \theta}
$$

as long as the sign convention is checked.

## 6. Motor, Spool, And TSA Sizing Chain

Motor or actuator sizing should be derived after estimating tendon tension and required pull stroke. The routing model should first stay actuator-agnostic and report:

- required tendon tension
- required tendon pull stroke
- required pull speed later, once a prescribed time trajectory exists

| Quantity | Formula / role | Notes |
|---|---|---|
| Tendon pull distance | $s_{\mathrm{pull}}(q)$ | from routing path length, with correct sign |
| Spool radius | $r_{\mathrm{spool}}$ | required for spool-driven motor sizing |
| Spool rotation | $\theta_{\mathrm{spool}} = s_{\mathrm{pull}} / r_{\mathrm{spool}}$ | assumes no slip and constant radius |
| Tendon speed | $\dot{s} = - (\partial L/\partial q)\dot{q}$ | sign depends on flexion/extension path |
| Spool speed | $\omega_{\mathrm{spool}} = \dot{s} / r_{\mathrm{spool}}$ | actuator velocity requirement |
| Spool torque | $\tau_{\mathrm{spool}} = T_{\mathrm{tendon}} r_{\mathrm{spool}} / \eta_{\mathrm{spool}}$ | output-side torque |
| Motor torque | $\tau_{\mathrm{motor}} = \tau_{\mathrm{spool}} / (G\eta_{\mathrm{gear}})$ | if gearbox ratio $G$ multiplies motor torque at the spool |
| Motor speed | $\omega_{\mathrm{motor}} = G\omega_{\mathrm{spool}}$ | speed/torque tradeoff |

For a twisted string actuator, the mapping from twist to tendon contraction is nonlinear and depends on string length, radius, twist angle, and allowable twist. Therefore TSA feasibility should be checked after the routing model reports required stroke and tension. TSA can remain the preferred actuator candidate, but it should be treated as a compact tendon displacement/force source, not as variable stiffness by itself.

Stroke feasibility is initially a provisional design constraint. Before actuator selection, the model should track `stroke_required_mm` as an output and reject only candidates that are clearly outside a compact-device budget. After actuator selection, the allowable stroke should be replaced by a spool-, TSA-, or linear-actuator-specific limit.

Near-term actuator outputs to track:

- peak tendon tension
- continuous/average tendon tension over a cycle
- maximum required tendon stroke
- maximum pull speed, once $q(t)$ is prescribed
- required spool torque or TSA operating range after choosing an actuator architecture

Do not over-model spool wall thickness yet. Do include spool radius for spool-driven concepts because it directly changes motor torque, speed, and revolutions required.

## 7. Trajectories, Speed, Rhythm, And Sequence Effect

The current routing model gives geometry-dependent movement relationships, not a full time trajectory by itself. To simulate speed and rhythm, impose or solve a time-varying joint trajectory/input.

Simple prescribed trajectory:

$$
\begin{aligned}
q_{\mathrm{des}}(t) &= q_{\mathrm{mean}} + A\sin(2\pi ft + \varphi) \\
\dot{q}_{\mathrm{des}}(t) &= 2\pi f A\cos(2\pi ft + \varphi)
\end{aligned}
$$

Coordinated index-finger trajectory:

$$
\begin{aligned}
\theta_{\mathrm{PIP}}(t) &= \beta_{\mathrm{PIP}}\theta_{\mathrm{MCP}}(t) \\
\theta_{\mathrm{DIP}}(t) &= \beta_{\mathrm{DIP}}\theta_{\mathrm{PIP}}(t)
\end{aligned}
$$

Tapping-like outputs:

| Output | Sim definition | Experiment measurement |
|---|---|---|
| Angular amplitude | peak-to-peak $q(t)$ | marker/video joint angles |
| Fingertip amplitude | peak-to-peak $p_{\mathrm{tip}}(t)$ or distance to pad | marker/video fingertip position |
| Speed | peak $\dot{q}$, peak $v_{\mathrm{tip}}$, or cycle time | video/encoder derivative |
| Rhythm | cycle period variability | timing of peaks/contacts |
| Sequence effect | amplitude decay across repeated cycles | slope of peak amplitude over taps |

These long-term outputs require a dynamic or commanded-input layer. They should not be claimed from the current quasi-static routing model alone.


## 8. Flexion, Extension, And Branched Inputs

Use the same structure for flexion and extension inputs, but keep independent tension inputs distinct from geometric branches.

For one flexion input with two branches:

$$
\begin{aligned}
R_{\mathrm{flex,total}}(q) &= R_{\mathrm{flex,left}}(q) + R_{\mathrm{flex,right}}(q) \\
\tau_{\mathrm{flex}}(q,T_{\mathrm{flex}}) &= T_{\mathrm{flex}}R_{\mathrm{flex,total}}(q)
\end{aligned}
$$

For a later extension input with two branches:

$$
\begin{aligned}
R_{\mathrm{ext,total}}(q) &= R_{\mathrm{ext,left}}(q) + R_{\mathrm{ext,right}}(q) \\
\tau_{\mathrm{ext}}(q,T_{\mathrm{ext}}) &= T_{\mathrm{ext}}R_{\mathrm{ext,total}}(q)
\end{aligned}
$$

Possible architecture stages:

| Stage | Actuation assumption | Why |
|---|---|---|
| V0 | one flexion input split into one or two side branches + passive return | simplest routing and motor/TSA sizing baseline |
| V1 | flexion input + dorsal elastic extension return | useful for tapping repetition without a second driven input |
| V2 | one flexion input and one extension input, each possibly branched | true bidirectional control and better stiffness modulation, but more slack/routing complexity |
| V3 | antagonistic co-tension with controlled stiffness | strongest variable-stiffness interpretation, but likely after v0 validation |

If variable stiffness comes from antagonistic co-tension, extension cannot remain an afterthought. If variable stiffness comes from a series elastic or adjustable transmission element, extension can stay passive longer.

## 9. Simulation Outputs Versus Experiment Outputs

| Model output | Compare to experiment? | Practical measurement |
|---|---|---|
| Joint angles $q$ | yes | video markers, fiducials, or encoderized joints |
| Fingertip position $p_{\mathrm{tip}}$ | yes | video/marker tracking |
| Tendon pull/stroke $s_{\mathrm{pull}}$ | yes | motor encoder, TSA twist estimate, spool rotation, or inline displacement |
| Tendon tension $T$ | yes if possible | load cell or calibrated motor current estimate |
| Passive torque curve | yes | manual displacement with load/torque estimate |
| Equilibrium pose for given pull/tension | yes | static hold tests |
| Branch balance | yes if branched | compare left/right branch displacement, slack, or visible motion symmetry |
| Ring placement sensitivity | partly | repeated tests with ring positions moved |
| Ring diameter/wall thickness | not directly as model validation | fit, comfort, deformation, print success |
| Friction/slack/hysteresis | later | load-unload cycles and motion lag |
| Speed/rhythm/sequence effect | later | repeated tapping trajectories |

First validation target: compare simulated and measured tendon stroke, joint posture, and fingertip position for a few static or slow quasi-static poses. Add speed/rhythm only after the static model is not obviously wrong.

Minimum experimental collection list for the first bench-top rig:

- tendon pull displacement
- tendon tension
- actuator command, motor encoder, or TSA twist angle
- motor current and voltage if available
- MCP/PIP/DIP joint angles from video or markers
- fingertip trajectory
- loading-unloading curve for hysteresis
- cycle-to-cycle repeatability
- slack or dead-zone before motion begins



## 10. Free-Body Diagram Description

A useful FBD for the current model would show each phalanx as a rigid body connected by revolute joints:

- MCP, PIP, and DIP joint reaction forces at the joint centers.
- Passive torsional spring/damper torque at each joint, initially spring only.
- Tendon force along each straight tendon segment between routing points.
- Guide/ring reaction forces where the tendon changes direction.
- Distal thimble anchor force where the tendon terminates.
- Optional fingertip contact force against a thumb pad or instrumented surface.
- Gravity and inertia omitted in the first quasi-static model, then added only if experiments show they matter.

For equations, virtual work is cleaner than drawing every guide reaction. The tendon contribution to joint torque can be computed from $-T\,\partial L / \partial q$ without explicitly solving all guide reaction forces.


## 11. What Is Correct, What Needs Correction

On the right track:

- Starting from phalanx lengths, joint angles, ROM, passive stiffness, tendon routing, ring placement, and thimble anchor is correct.
- Treating ring count, branch count, and ring/guide placement as design variables is correct.
- Separating flexion and extension paths is correct, even if extension is passive in the first prototype.
- Using $L(q)$, $\Delta L$, and $\partial L / \partial q$ is the right mathematical backbone.
- Thinking about motor torque now is useful, as long as it is downstream of tendon tension and spool radius.

Needs correction or tightening:

- $\tau_{\mathrm{tendon}}$ is a joint-torque vector, not the motor torque. Motor torque comes later through spool radius, gearing, and efficiency.
- $T_{\mathrm{req}} = \tau_{\mathrm{tendon}} / \lvert \partial L / \partial q \rvert$ only works cleanly for one joint. For three joints and one independent tension input, use equilibrium solving or least-squares projection onto the total branch moment-arm map.
- Do not collapse actuator count, independent tension inputs, and routing branch count into one phrase. A single input can have multiple branches.
- Passive torque should include a sign convention. $k(q-q_0)$ is the required holding torque; the passive resisting torque is usually the negative of that.
- Speed, rhythm, and sequence effect require a time trajectory or dynamic/control layer. The routing model alone only gives the geometry needed to compute stroke and velocity from a prescribed $q(t)$.
- Ring diameter, wall thickness, and slot diameter are not primary physics outputs yet. They are CAD/fabrication constraints informed by finger size, tendon diameter, comfort, and strength.
- Spool thickness can wait, but spool radius cannot wait if motor torque and speed are being estimated.
- Friction, slack, tendon stretch, branch imbalance, splitter/yoke behavior, and ring deformation can be omitted first, but they should be listed as known limitations because they may dominate the real prototype.


## 12. Recommended Next Questions

1. What is the first target motion: generic curl, pinch closure against a fixed thumb pad, or repetitive tapping-like flexion/extension?
2. Is the first prototype flexion-only with passive return, flexion plus elastic return, or true antagonistic flexion/extension?
3. Which variables should be normalized by finger length in both Python and Fusion: ring longitudinal position, anchor position, and tendon offsets?
4. What are acceptable error metrics for the first validation: joint angle error, fingertip-position error, tendon-stroke error, or tension error?
5. How much underactuation mismatch is acceptable before adding another guide, changing routing side, or adding an extension path?
6. Which measurements are realistic in the first bench test: video markers, motor encoder, load cell, current sensing, or all of them?
7. Should passive stiffness represent the human finger alone, the device alone, or the combined finger-device system?
8. What minimum motor-sizing assumptions are acceptable now: tendon tension estimate, spool radius or TSA geometry, gear ratio, efficiency, and desired cycle speed?
9. If one input splits into left/right branches, how will the splitter or yoke equalize tension and prevent one branch from going slack?
10. Is variable stiffness being claimed at the actuator, tendon/transmission, joint, or fingertip/end-effector level?

Recommended next implementation step: add a small Python function that takes one or more routing branches sharing an input and returns $p_{\mathrm{tip}}(q)$, branch lengths $L_j(q)$, actuator-equivalent pull stroke, total moment-arm map $R_{\mathrm{total}}(q)$, torque per newton of shared tendon tension, and a least-squares $T_{\mathrm{req}}$ estimate for a chosen passive torque law. Then rank one-branch versus two-branch flexion routes before designing fixed CAD details.
